# 👗 AI Virtual Outfit Try-On System

An AI-powered virtual try-on experience that works entirely in Google Colab.
The system uses **MediaPipe Pose** to detect your body landmarks and overlays
selected clothing items in real time.

---
**How to use:**
1. Run **Cell 1** to install dependencies
2. Run **Cell 2** to clone the helper utilities
3. Run **Cell 3** to launch the interactive UI
4. Click **▶ Start Camera**, choose a garment, then click **📸 Screenshot**


In [ ]:
# Cell 1 – Install dependencies
print('Installing dependencies – this may take a minute…')
import subprocess, sys
pkgs = [
    'opencv-python-headless',
    'mediapipe',
    'Pillow',
    'numpy',
    'ipywidgets',
    'matplotlib',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('✅ All dependencies installed.')


In [ ]:
# Cell 2 – Clone the repository so utility modules are available
import os, sys

REPO_URL  = 'https://github.com/Momina-13/AI-Virtual-outfit-Try-on.git'
REPO_DIR  = '/content/AI-Virtual-outfit-Try-on'

if not os.path.isdir(REPO_DIR):
    os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    print('✅ Repository cloned.')
else:
    print('ℹ️  Repository already present.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    print('✅ Added to Python path.')

SAMPLES_DIR = os.path.join(REPO_DIR, 'clothing_samples')
print(f'Clothing samples found: {os.listdir(SAMPLES_DIR)}')


In [ ]:
# Cell 3 – AI Virtual Outfit Try-On interactive UI
import os, sys, time, base64, io, threading
from datetime import datetime

import cv2
import numpy as np
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

# Make sure the repo root is on the path
REPO_DIR = '/content/AI-Virtual-outfit-Try-on'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from utils.pose_detector   import PoseDetector
from utils.clothing_overlay import ClothingOverlay
from utils.camera_utils    import CameraUtils

# ---------------------------------------------------------------------------
# JavaScript: initialise webcam and expose frame-capture helper
# ---------------------------------------------------------------------------
CAMERA_JS = """
window._tryOnActive = false;

async function startTryOnCamera(width, height) {
  if (window._tryOnStream) return 'already_running';
  const stream = await navigator.mediaDevices.getUserMedia(
      { video: { width, height } });
  window._tryOnStream = stream;
  const v = document.createElement('video');
  v.srcObject = stream;
  v.autoplay = true;
  v.playsInline = true;
  await new Promise(r => { v.onloadedmetadata = r; });
  v.play();
  window._tryOnVideo = v;
  window._tryOnActive = true;
  return 'started';
}

function captureTryOnFrame(width, height) {
  const v = window._tryOnVideo;
  if (!v) return null;
  const c = document.createElement('canvas');
  c.width  = v.videoWidth  || width;
  c.height = v.videoHeight || height;
  c.getContext('2d').drawImage(v, 0, 0);
  return c.toDataURL('image/jpeg', 0.85);
}

function stopTryOnCamera() {
  if (!window._tryOnStream) return 'not_running';
  window._tryOnStream.getTracks().forEach(t => t.stop());
  delete window._tryOnStream;
  delete window._tryOnVideo;
  window._tryOnActive = false;
  return 'stopped';
}
"""

display(Javascript(CAMERA_JS))

# ---------------------------------------------------------------------------
# Build clothing catalog
# ---------------------------------------------------------------------------
SAMPLES_DIR = os.path.join(REPO_DIR, 'clothing_samples')
catalog = ClothingOverlay.list_catalog(SAMPLES_DIR)
catalog_paths = {name: os.path.join(SAMPLES_DIR, name) for name in catalog}

# Pre-load clothing overlays
overlays = {name: ClothingOverlay(path) for name, path in catalog_paths.items()}

# ---------------------------------------------------------------------------
# Pose detector
# ---------------------------------------------------------------------------
detector = PoseDetector(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# ---------------------------------------------------------------------------
# Widget layout
# ---------------------------------------------------------------------------
FRAME_W, FRAME_H = 640, 480

clothing_dd = widgets.Dropdown(
    options=['(none)'] + catalog,
    value='(none)',
    description='Garment:',
    layout=widgets.Layout(width='250px'),
)

x_offset_sl = widgets.IntSlider(
    value=0, min=-100, max=100, step=5,
    description='X offset:',
    layout=widgets.Layout(width='350px'),
)
y_offset_sl = widgets.IntSlider(
    value=0, min=-100, max=100, step=5,
    description='Y offset:',
    layout=widgets.Layout(width='350px'),
)
scale_sl = widgets.FloatSlider(
    value=1.2, min=0.8, max=2.0, step=0.05,
    description='Scale:',
    layout=widgets.Layout(width='350px'),
)
conf_sl = widgets.FloatSlider(
    value=0.5, min=0.1, max=0.95, step=0.05,
    description='Min conf:',
    layout=widgets.Layout(width='350px'),
)

start_btn = widgets.Button(
    description='▶ Start Camera',
    button_style='success',
    layout=widgets.Layout(width='160px'),
)
stop_btn = widgets.Button(
    description='⏹ Stop Camera',
    button_style='danger',
    layout=widgets.Layout(width='160px'),
)
screenshot_btn = widgets.Button(
    description='📸 Screenshot',
    button_style='info',
    layout=widgets.Layout(width='160px'),
)

status_lbl  = widgets.Label(value='Camera stopped.')
frame_out   = widgets.Output()
screenshot_out = widgets.Output()

# ---------------------------------------------------------------------------
# State
# ---------------------------------------------------------------------------
state = {
    'running': False,
    'latest_frame': None,   # BGR ndarray
}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def data_url_to_bgr(data_url):
    _, encoded = data_url.split(',', 1)
    pil = Image.open(io.BytesIO(base64.b64decode(encoded))).convert('RGB')
    return cv2.cvtColor(np.array(pil, dtype=np.uint8), cv2.COLOR_RGB2BGR)


def process_frame(bgr):
    """Run pose detection and clothing overlay, return annotated BGR frame."""
    chosen = clothing_dd.value
    landmarks, annotated = detector.detect(bgr)
    if landmarks and detector.is_pose_visible(landmarks, conf_sl.value):
        measurements = detector.get_body_measurements(landmarks, bgr.shape)
        if chosen != '(none)' and chosen in overlays:
            annotated = overlays[chosen].apply(
                annotated, measurements,
                x_offset=x_offset_sl.value,
                y_offset=y_offset_sl.value,
                scale_factor=scale_sl.value,
            )
    return annotated


def render_bgr(bgr):
    """Convert BGR to base64 JPEG and embed as an img tag."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    buf = io.BytesIO()
    Image.fromarray(rgb).save(buf, format='JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<img src="data:image/jpeg;base64,{b64}" style="max-width:100%;"/>'

# ---------------------------------------------------------------------------
# Camera loop (runs in a background thread)
# ---------------------------------------------------------------------------

def camera_loop():
    from google.colab.output import eval_js  # type: ignore
    js_capture = f'captureTryOnFrame({FRAME_W}, {FRAME_H})'
    while state['running']:
        try:
            data_url = eval_js(js_capture)
            if not data_url:
                time.sleep(0.05)
                continue
            bgr = data_url_to_bgr(data_url)
            result = process_frame(bgr)
            state['latest_frame'] = result
            with frame_out:
                clear_output(wait=True)
                display(HTML(render_bgr(result)))
        except Exception as exc:
            status_lbl.value = f'⚠️ Error: {exc}'
            time.sleep(0.1)

# ---------------------------------------------------------------------------
# Button handlers
# ---------------------------------------------------------------------------

def on_start(_):
    if state['running']:
        return
    from google.colab.output import eval_js  # type: ignore
    status = eval_js(f'startTryOnCamera({FRAME_W}, {FRAME_H})')
    status_lbl.value = f'Camera {status}. Detecting pose…'
    state['running'] = True
    t = threading.Thread(target=camera_loop, daemon=True)
    t.start()


def on_stop(_):
    state['running'] = False
    from google.colab.output import eval_js  # type: ignore
    eval_js('stopTryOnCamera()')
    status_lbl.value = 'Camera stopped.'
    with frame_out:
        clear_output()


def on_screenshot(_):
    frame = state.get('latest_frame')
    if frame is None:
        with screenshot_out:
            clear_output(wait=True)
            print('⚠️ No frame available – start the camera first.')
        return
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    fname = f'virtual_outfit_{ts}.jpg'
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    buf = io.BytesIO()
    Image.fromarray(rgb).save(buf, format='JPEG', quality=95)
    buf.seek(0)
    # Save locally and offer download via Colab
    with open(fname, 'wb') as f:
        f.write(buf.getvalue())
    b64 = base64.b64encode(buf.getvalue()).decode()
    dl_html = (
        f'<b>📸 Screenshot saved as <code>{fname}</code></b><br/>'
        f'<a href="data:image/jpeg;base64,{b64}" download="{fname}">'
        f'⬇️ Download {fname}</a><br/>'
        f'<img src="data:image/jpeg;base64,{b64}" style="max-width:400px;"/>'
    )
    with screenshot_out:
        clear_output(wait=True)
        display(HTML(dl_html))


start_btn.on_click(on_start)
stop_btn.on_click(on_stop)
screenshot_btn.on_click(on_screenshot)

# ---------------------------------------------------------------------------
# Layout and display
# ---------------------------------------------------------------------------
controls = widgets.VBox([
    widgets.HTML('<h2>👗 AI Virtual Outfit Try-On</h2>'),
    widgets.HBox([start_btn, stop_btn, screenshot_btn]),
    clothing_dd,
    widgets.HTML('<b>Fine-tune position & scale:</b>'),
    x_offset_sl,
    y_offset_sl,
    scale_sl,
    conf_sl,
    status_lbl,
])

display(widgets.HBox([
    controls,
    widgets.VBox([frame_out]),
]))
display(screenshot_out)


In [ ]:
# Cell 4 – Offline demo: test pose detection + clothing overlay on a static image
# (No camera required – useful for testing outside Colab)
import os, sys

REPO_DIR = '/content/AI-Virtual-outfit-Try-on'
if not os.path.isdir(REPO_DIR):
    REPO_DIR = os.path.dirname(os.path.abspath('.'))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

from utils.clothing_overlay import ClothingOverlay

SAMPLES_DIR = os.path.join(REPO_DIR, 'clothing_samples')
catalog = ClothingOverlay.list_catalog(SAMPLES_DIR)
print('Available clothing items:', catalog)

# Display thumbnails of the catalog
fig, axes = plt.subplots(1, len(catalog), figsize=(4 * len(catalog), 5))
if len(catalog) == 1:
    axes = [axes]
for ax, name in zip(axes, catalog):
    img = Image.open(os.path.join(SAMPLES_DIR, name)).convert('RGBA')
    ax.imshow(img)
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Clothing Catalog', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Catalog loaded successfully. Run Cell 3 for the live try-on experience.')
